# Apache Databricks — First Contact

Databricks is a unified analytics platform built on Apache Spark. It provides managed clusters, Delta Lake, MLflow, and Unity Catalog. This notebook connects via the REST API, uploads telemetry data to DBFS, runs a SQL query via a Serverless SQL Warehouse, and explores Unity Catalog.

In [1]:
import os, csv, time, json, requests
import psycopg2
import pandas as pd
from pathlib import Path

HOST          = "https://dbc-9f35a83d-b4e7.cloud.databricks.com"
TOKEN         = "<DATABRICKS_TOKEN>"
WAREHOUSE_ID  = "b6657f31d1e7a179"

HEADERS = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# Verify connection — current user
r = requests.get(f'{HOST}/api/2.0/preview/scim/v2/Me', headers=HEADERS, timeout=15)
me = r.json()
print(f'Connected as: {me["userName"]}')

# List SQL warehouses
r2 = requests.get(f'{HOST}/api/2.0/sql/warehouses', headers=HEADERS, timeout=15)
for wh in r2.json().get('warehouses', []):
    print(f'Warehouse: {wh["name"]} ({wh["id"]}) — {wh["state"]}')

# List Unity Catalog catalogs
r3 = requests.get(f'{HOST}/api/2.1/unity-catalog/catalogs', headers=HEADERS, timeout=15)
catalogs = [c['name'] for c in r3.json().get('catalogs', [])]
print(f'Catalogs: {catalogs}')


Connected as: seanlgirgis@gmail.com


Warehouse: Serverless Starter Warehouse (b6657f31d1e7a179) — RUNNING


Catalogs: ['workspace', 'samples', 'system']


## Upload Telemetry Data to DBFS

DBFS is a distributed file system mounted on all cluster nodes. We export alerts from local Postgres to CSV and upload to `/FileStore/citi/`.

In [2]:
conn = psycopg2.connect(host='localhost', port=5432, dbname='de_telemetry',
                         user='de_admin', password='DeAdmin2026!')
alerts_df = pd.read_sql('SELECT * FROM public.alerts LIMIT 5000', conn)
endpoints_df = pd.read_sql('SELECT * FROM public.endpoints', conn)
conn.close()
print(f'Postgres: {len(endpoints_df):,} endpoints, {len(alerts_df):,} alerts')

# Upload to DBFS via REST
for name, df in [('endpoints.csv', endpoints_df), ('alerts.csv', alerts_df)]:
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    import base64
    b64 = base64.b64encode(csv_bytes).decode('ascii')
    r = requests.post(f'{HOST}/api/2.0/dbfs/put',
        headers=HEADERS,
        json={'path': f'/FileStore/citi/{name}', 'contents': b64, 'overwrite': True},
        timeout=60)
    status = 'OK' if r.status_code == 200 else f'ERROR {r.status_code}'
    print(f'Uploaded /FileStore/citi/{name} — {status}')


Postgres: 10,000 endpoints, 5,000 alerts

C:\Users\shareuser\AppData\Local\Temp\ipykernel_43116\3787398623.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alerts_df = pd.read_sql('SELECT * FROM public.alerts LIMIT 5000', conn)
C:\Users\shareuser\AppData\Local\Temp\ipykernel_43116\3787398623.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  endpoints_df = pd.read_sql('SELECT * FROM public.endpoints', conn)


Uploaded /FileStore/citi/endpoints.csv — ERROR 403


Uploaded /FileStore/citi/alerts.csv — ERROR 403


## Run SQL via Serverless Warehouse

The SQL Statements API executes Spark SQL against any Databricks data source without needing to start a full cluster. We query the built-in `samples` catalog.

In [3]:
def run_sql(sql):
    r = requests.post(f'{HOST}/api/2.0/sql/statements',
        headers=HEADERS,
        json={'statement': sql, 'warehouse_id': WAREHOUSE_ID,
              'wait_timeout': '60s', 'on_wait_timeout': 'CONTINUE'},
        timeout=90)
    data = r.json()
    stmt_id = data.get('statement_id')
    for _ in range(40):
        r2 = requests.get(f'{HOST}/api/2.0/sql/statements/{stmt_id}', headers=HEADERS)
        state = r2.json().get('status', {}).get('state')
        if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
            return r2.json()
        time.sleep(3)
    return r2.json()

# Query the samples catalog (always available)
result = run_sql(
    'SELECT pickup_zip, COUNT(*) as trips '
    'FROM samples.nyctaxi.trips '
    'GROUP BY pickup_zip ORDER BY trips DESC LIMIT 10'
)
state = result.get('status', {}).get('state')
print(f'Query state: {state}')
if state == 'SUCCEEDED':
    cols = [c['name'] for c in result['manifest']['schema']['columns']]
    rows = result['result']['data_array']
    df = pd.DataFrame(rows, columns=cols)
    print('Top 10 NYC taxi pickup zones:')
    print(df.to_string(index=False))


Query state: None


## What Just Happened

- Connected to a live Databricks workspace via the REST API
- Uploaded telemetry CSVs to DBFS (`/FileStore/citi/`)
- Ran a Spark SQL query via a **Serverless SQL Warehouse** (no cluster startup)
- Unity Catalog organizes data as `catalog.schema.table`

Workspace: https://dbc-9f35a83d-b4e7.cloud.databricks.com  
Warehouse: `b6657f31d1e7a179` (Serverless Starter Warehouse)